# Masterclass Notebook: Complete Linear Regression Guide for ML Associates

This Jupyter Notebook accompanies the master guide on **Linear Regression**. It contains complete theoretical breakdowns, mathematical derivations, diagnostic tests, regularization paths, feature engineering, and runnable Python examples for every core aspect.

---


In [ ]:
# Imports & Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

from sklearn.datasets import fetch_california_housing, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 10
warnings.filterwarnings("ignore")

print("Notebook environment ready! All libraries imported successfully.")


## 1. Introduction & Fundamental Concepts

**Linear Regression** is a fundamental supervised learning algorithm used to model the linear relationship between one or more independent predictor variables ($X$) and a continuous target variable ($y$).

### Classification of Linear Regression
- **Simple Linear Regression**: Uses a single feature $x$ to predict target $y$.
- **Multiple Linear Regression**: Uses multiple features $x_1, x_2, \dots, x_p$ to predict target $y$.
- **Multivariate Linear Regression**: Predicts multiple continuous target variables $y_1, y_2, \dots, y_k$ simultaneously (less common, distinct from multiple linear regression).


In [ ]:
# Example 1: Simple vs Multiple Linear Regression Data Visualizations
np.random.seed(42)

# Simple Linear Regression (1 Feature)
X_simple = 2 * np.random.rand(100, 1)
y_simple = 4 + 3 * X_simple.squeeze() + np.random.randn(100) * 0.8

# Multiple Linear Regression (2 Features for 3D visualization)
x1_mult = np.random.uniform(-5, 5, 100)
x2_mult = np.random.uniform(-5, 5, 100)
y_mult = 3 + 2*x1_mult - 1.5*x2_mult + np.random.randn(100) * 2

fig = plt.figure(figsize=(14, 5))

# Plot 1: Simple Linear Regression
ax1 = fig.add_subplot(1, 2, 1)
ax1.scatter(X_simple, y_simple, color='dodgerblue', alpha=0.7, label='Observed Data')
ax1.plot(X_simple, 4 + 3*X_simple, color='crimson', linewidth=2, label='Regression Line (y = 4 + 3x)')
ax1.set_title("1. Simple Linear Regression (1 Predictor)")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.legend()

# Plot 2: Multiple Linear Regression (Hyperplane)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.scatter(x1_mult, x2_mult, y_mult, color='teal', alpha=0.8)
ax2.set_title("2. Multiple Linear Regression Plane (2 Predictors)")
ax2.set_xlabel("Feature x1")
ax2.set_ylabel("Feature x2")
ax2.set_zlabel("Target y")

plt.tight_layout()
plt.show()


## 2. Mathematical Formulations

### Simple Linear Regression
$$y = \beta_0 + \beta_1 x + \epsilon$$

- $y$: Dependent target variable.
- $x$: Independent feature variable.
- $\beta_0$: $y$-intercept (value of $y$ when $x = 0$).
- $\beta_1$: Slope coefficient (change in $y$ per unit change in $x$).
- $\epsilon$: Error term (unexplained noise/residuals), where $\epsilon \sim \mathcal{N}(0, \sigma^2)$.

### Multiple Linear Regression (Scalar Form)
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p + \epsilon$$

### Multiple Linear Regression (Matrix / Vector Form)
For $n$ observations and $p$ features:

$$\mathbf{y} = \mathbf{X} \boldsymbol{\beta} + \boldsymbol{\epsilon}$$

Where:
- $\mathbf{y}$ is an $n \times 1$ target vector.
- $\mathbf{X}$ is an $n \times (p+1)$ design matrix (including a column of $1$s for the intercept $\beta_0$):
  $$\mathbf{X} = \begin{bmatrix} 1 & x_{11} & x_{12} & \dots & x_{1p} \\ 1 & x_{21} & x_{22} & \dots & x_{2p} \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 1 & x_{n1} & x_{n2} & \dots & x_{np} \end{bmatrix}$$
- $\boldsymbol{\beta}$ is a $(p+1) \times 1$ parameter/weight vector: $\boldsymbol{\beta} = [\beta_0, \beta_1, \dots, \beta_p]^T$.
- $\boldsymbol{\epsilon}$ is an $n \times 1$ residual error vector.


In [ ]:
# Example 2: Matrix & Vector Formulations in NumPy
np.random.seed(42)
n_samples, p_features = 5, 3

# Raw Feature Matrix (5 samples, 3 features)
X_raw = np.random.randint(1, 10, size=(n_samples, p_features))

# Design Matrix X (adding column of 1s for intercept beta_0)
X_design = np.c_[np.ones((n_samples, 1)), X_raw]

# True Parameter Vector beta = [beta_0, beta_1, beta_2, beta_3]^T
true_beta = np.array([2.5, 1.0, -0.5, 3.0]).reshape(-1, 1)

# Residual noise vector epsilon ~ N(0, 0.1^2)
epsilon = np.random.randn(n_samples, 1) * 0.1

# Compute target vector y = X * beta + epsilon
y_vector = np.dot(X_design, true_beta) + epsilon

print("=== DESIGN MATRIX X (n x (p+1)) ===")
print(X_design)
print("
=== PARAMETER VECTOR beta ((p+1) x 1) ===")
print(true_beta)
print("
=== RESULTING TARGET VECTOR y (n x 1) ===")
print(y_vector.round(3))


## 3. The 5 Core Assumptions of Linear Regression & Diagnostics

For Ordinary Least Squares (OLS) estimates to be **BLUE** (**Best Linear Unbiased Estimator**) according to the **Gauss-Markov Theorem**, the dataset and model residuals must satisfy five fundamental assumptions:

| Assumption | Diagnostic Tool | Remedial Action |
| :--- | :--- | :--- |
| **1. Linearity** | Residuals vs. Fitted plot | Log/Polynomial transforms |
| **2. Independence** | Durbin-Watson Test (~2.0) | Time-series / AR models |
| **3. Homoscedasticity** | Breusch-Pagan / Scale-Location | Box-Cox / Log transform |
| **4. Normality** | Q-Q Plot / Shapiro-Wilk Test | Non-linear transform |
| **5. No Multicollinearity** | Variance Inflation Factor (VIF) | Drop features / Ridge |


In [ ]:
# Example 3.1: Linearity Diagnostic (Residuals vs Fitted Plot)
np.random.seed(42)
x_lin = np.linspace(-3, 3, 100)

# 1. Non-linear Data (y = x^2 + noise)
y_nonlinear = x_lin**2 + np.random.randn(100) * 0.5
model_nl = LinearRegression().fit(x_lin.reshape(-1, 1), y_nonlinear)
res_nl = y_nonlinear - model_nl.predict(x_lin.reshape(-1, 1))

# 2. Linear Data (y = 2x + noise)
y_linear = 2*x_lin + np.random.randn(100) * 0.5
model_l = LinearRegression().fit(x_lin.reshape(-1, 1), y_linear)
res_l = y_linear - model_l.predict(x_lin.reshape(-1, 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Non-Linear Violation (U-Shape Pattern in Residuals)
axes[0].scatter(model_nl.predict(x_lin.reshape(-1, 1)), res_nl, color='crimson', alpha=0.7)
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title("VIOLATION: Non-Linearity (U-Shape Residual Pattern)")
axes[0].set_xlabel("Fitted Values")
axes[0].set_ylabel("Residuals")

# Satisfied Linearity (Random Noise around 0)
axes[1].scatter(model_l.predict(x_lin.reshape(-1, 1)), res_l, color='seagreen', alpha=0.7)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_title("SATISFIED: Linearity (Random Noise Pattern)")
axes[1].set_xlabel("Fitted Values")
axes[1].set_ylabel("Residuals")

plt.tight_layout()
plt.show()


In [ ]:
# Example 3.2: Independence of Errors (Durbin-Watson Test)
# Durbin-Watson d = sum( (e_t - e_{t-1})^2 ) / sum( e_t^2 )
# d ~ 2: No autocorrelation | d < 1.5: Positive autocorrelation | d > 2.5: Negative autocorrelation

np.random.seed(42)
t = np.arange(100)

# Autocorrelated Errors (Random Walk noise)
autocorr_errors = np.cumsum(np.random.randn(100) * 0.5)
y_autocorr = 2 + 0.5*t + autocorr_errors

# Independent Errors
indep_errors = np.random.randn(100) * 2.0
y_indep = 2 + 0.5*t + indep_errors

res_auto = y_autocorr - LinearRegression().fit(t.reshape(-1, 1), y_autocorr).predict(t.reshape(-1, 1))
res_indep = y_indep - LinearRegression().fit(t.reshape(-1, 1), y_indep).predict(t.reshape(-1, 1))

dw_auto = np.sum(np.diff(res_auto)**2) / np.sum(res_auto**2)
dw_indep = np.sum(np.diff(res_indep)**2) / np.sum(res_indep**2)

print(f"Durbin-Watson Statistic (Autocorrelated Errors): {dw_auto:.3f} -> (Violated: Positive Autocorrelation)")
print(f"Durbin-Watson Statistic (Independent Errors)   : {dw_indep:.3f} -> (Satisfied: Near 2.0)")


In [ ]:
# Example 3.3: Homoscedasticity vs Heteroscedasticity & Log Transformation Fix
np.random.seed(42)
x_het = np.linspace(1, 10, 150)

# Heteroscedastic data: error variance increases proportional to x (funnel shape)
noise_het = np.random.randn(150) * x_het * 0.8
y_het = 3 * x_het + noise_het

# Model on original target
model_het = LinearRegression().fit(x_het.reshape(-1, 1), y_het)
y_pred_het = model_het.predict(x_het.reshape(-1, 1))
res_het = y_het - y_pred_het

# Fix: Log transformation on positive y
y_pos = y_het - y_het.min() + 1 # shift to positive
y_log = np.log(y_pos)
model_log = LinearRegression().fit(x_het.reshape(-1, 1), y_log)
res_log = y_log - model_log.predict(x_het.reshape(-1, 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Funnel Shape Plot
axes[0].scatter(y_pred_het, res_het, color='darkorange', alpha=0.7)
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title("HETEROSCEDASTICITY: Funnel/Cone Residual Shape")
axes[0].set_xlabel("Fitted Values")
axes[0].set_ylabel("Residuals")

# Log Transformed Homoscedastic Plot
axes[1].scatter(model_log.predict(x_het.reshape(-1, 1)), res_log, color='dodgerblue', alpha=0.7)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_title("FIXED via Log Transformation: Equal Variance")
axes[1].set_xlabel("Fitted Values (Log Scale)")
axes[1].set_ylabel("Residuals")

plt.tight_layout()
plt.show()


In [ ]:
# Example 3.4: Normality of Residuals (Q-Q Plot & Shapiro-Wilk Test)
np.random.seed(42)

# Normal Residuals vs Non-Normal Skewed Residuals
norm_res = np.random.normal(0, 1, 100)
skewed_res = np.random.exponential(1.5, 100) - 1.5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Normal Q-Q Plot
stats.probplot(norm_res, dist="norm", plot=axes[0])
axes[0].set_title("SATISFIED: Normal Residual Q-Q Plot")

# Skewed Q-Q Plot
stats.probplot(skewed_res, dist="norm", plot=axes[1])
axes[1].set_title("VIOLATED: Heavy-Tailed Skewed Residual Q-Q Plot")

plt.tight_layout()
plt.show()

# Shapiro-Wilk Normality Test (p > 0.05 indicates normal distribution)
stat_n, p_n = stats.shapiro(norm_res)
stat_s, p_s = stats.shapiro(skewed_res)

print(f"Shapiro-Wilk Test (Normal Residuals): p-value = {p_n:.4f} -> (Normally Distributed)")
print(f"Shapiro-Wilk Test (Skewed Residuals): p-value = {p_s:.4f} -> (Violates Normality)")


In [ ]:
# Example 3.5: Multicollinearity & Variance Inflation Factor (VIF)
# VIF_j = 1 / (1 - R_j^2)

np.random.seed(42)
x1 = np.random.randn(100)
x2 = x1 * 0.98 + np.random.randn(100) * 0.05 # 98% correlation with x1
x3 = np.random.randn(100)                    # Uncorrelated

df_vif = pd.DataFrame({"x1_Income": x1, "x2_Savings": x2, "x3_Age": x3})

def compute_vif_table(df):
    vif_list = []
    for col in df.columns:
        y_c = df[col]
        X_c = df.drop(columns=[col])
        r2_c = r2_score(y_c, LinearRegression().fit(X_c, y_c).predict(X_c))
        vif = 1.0 / (1.0 - r2_c)
        vif_list.append({"Feature": col, "VIF": round(vif, 2), "Status": "HIGH Multicollinearity" if vif > 5 else "OK"})
    return pd.DataFrame(vif_list)

print("=== VARIANCE INFLATION FACTOR (VIF) ANALYSIS ===")
print(compute_vif_table(df_vif))


## 4. Model Estimation & Fitting Algorithms

### A. Ordinary Least Squares (OLS) Analytical Solution

OLS minimizes the Sum of Squared Residuals (SSR):

$$S(\boldsymbol{\beta}) = \sum_{i=1}^n (y_i - \hat{y}_i)^2 = (\mathbf{y} - \mathbf{X}\boldsymbol{\beta})^T (\mathbf{y} - \mathbf{X}\boldsymbol{\beta})$$

Expanding $S(\boldsymbol{\beta})$:
$$S(\boldsymbol{\beta}) = \mathbf{y}^T\mathbf{y} - 2\boldsymbol{\beta}^T\mathbf{X}^T\mathbf{y} + \boldsymbol{\beta}^T\mathbf{X}^T\mathbf{X}\boldsymbol{\beta}$$

Taking the derivative with respect to $\boldsymbol{\beta}$ and setting to $0$:
$$\frac{\partial S}{\partial \boldsymbol{\beta}} = -2\mathbf{X}^T\mathbf{y} + 2\mathbf{X}^T\mathbf{X}\boldsymbol{\beta} = 0$$

$$\mathbf{X}^T\mathbf{X}\boldsymbol{\beta} = \mathbf{X}^T\mathbf{y}$$

Solving for $\boldsymbol{\hat{\beta}}$ yields the **Normal Equation**:

$$\boldsymbol{\hat{\beta}} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

---

### B. Gradient Descent (Iterative Solution)

Used when the number of features $p$ is very large ($p > 10,000$) or for online/streaming data learning.

#### Cost Function (Mean Squared Error)
$$J(\boldsymbol{\beta}) = \frac{1}{2n} \sum_{i=1}^n (\mathbf{x}_i^T \boldsymbol{\beta} - y_i)^2 = \frac{1}{2n} \|\mathbf{X}\boldsymbol{\beta} - \mathbf{y}\|^2_2$$

#### Gradient Calculation
$$\nabla_{\boldsymbol{\beta}} J(\boldsymbol{\beta}) = \frac{1}{n} \mathbf{X}^T (\mathbf{X}\boldsymbol{\beta} - \mathbf{y})$$

#### Weight Update Rule
$$\boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} - \alpha \nabla_{\boldsymbol{\beta}} J(\boldsymbol{\beta}^{(t)})$$


In [ ]:
# Example 4.1: OLS Closed Form vs. Batch Gradient Descent from Scratch
class LinearRegressionCustom:
    def __init__(self, method="ols", lr=0.01, n_iters=1000):
        self.method = method
        self.lr = lr
        self.n_iters = n_iters
        self.beta = None
        self.costs = []

    def fit(self, X, y):
        n, p = X.shape
        X_b = np.c_[np.ones((n, 1)), X]
        y_b = y.reshape(-1, 1)

        if self.method == "ols":
            # Normal Equation: beta = (X^T * X)^(-1) * X^T * y
            self.beta = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y_b)
            
        elif self.method == "gd":
            self.beta = np.zeros((p + 1, 1))
            for i in range(self.n_iters):
                grad = (1 / n) * X_b.T.dot(X_b.dot(self.beta) - y_b)
                self.beta -= self.lr * grad
                cost = (1 / (2 * n)) * np.sum((X_b.dot(self.beta) - y_b) ** 2)
                self.costs.append(cost)

    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b.dot(self.beta)

# Test both methods
X_sim = 2 * np.random.rand(100, 2)
y_sim = 4 + 3*X_sim[:, 0] - 2*X_sim[:, 1] + np.random.randn(100) * 0.2

ols_model = LinearRegressionCustom(method="ols")
ols_model.fit(X_sim, y_sim)

gd_model = LinearRegressionCustom(method="gd", lr=0.1, n_iters=500)
gd_model.fit(X_sim, y_sim)

print("=== PARAMETER ESTIMATES COMPARISON ===")
print("True Parameters : beta_0=4.0, beta_1=3.0, beta_2=-2.0")
print(f"OLS Parameters  : beta_0={ols_model.beta[0][0]:.4f}, beta_1={ols_model.beta[1][0]:.4f}, beta_2={ols_model.beta[2][0]:.4f}")
print(f"GD  Parameters  : beta_0={gd_model.beta[0][0]:.4f}, beta_1={gd_model.beta[1][0]:.4f}, beta_2={gd_model.beta[2][0]:.4f}")


## 5. Model Evaluation Metrics

Assume $y_i$ is actual, $\hat{y}_i$ is predicted, and $\bar{y}$ is the mean of true values.

### 1. Mean Absolute Error (MAE)
$$\text{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \hat{y}_i|$$

### 2. Mean Squared Error (MSE)
$$\text{MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

### 3. Root Mean Squared Error (RMSE)
$$\text{RMSE} = \sqrt{\text{MSE}} = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2}$$

### 4. Coefficient of Determination ($R^2$ Score)
$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2}$$

### 5. Adjusted $R^2$ Score
$$\bar{R}^2 = 1 - \left[ \frac{(1 - R^2)(n - 1)}{n - p - 1} \right]$$


In [ ]:
# Example 5.1: Manual Calculation of Evaluation Metrics vs. sklearn
np.random.seed(42)
y_act = np.array([10.0, 20.0, 30.0, 40.0, 50.0])
y_prd = np.array([12.0, 18.0, 33.0, 39.0, 54.0])
n_obs = len(y_act)
p_feat = 1

# Formulas
mae_val = np.mean(np.abs(y_act - y_prd))
mse_val = np.mean((y_act - y_prd) ** 2)
rmse_val = np.sqrt(mse_val)
ss_res = np.sum((y_act - y_prd) ** 2)
ss_tot = np.sum((y_act - np.mean(y_act)) ** 2)
r2_val = 1 - (ss_res / ss_tot)
adj_r2_val = 1 - ((1 - r2_val) * (n_obs - 1) / (n_obs - p_feat - 1))

print("=== EVALUATION METRICS COMPARISON ===")
print(f"MAE     : Manual={mae_val:.4f} | sklearn={mean_absolute_error(y_act, y_prd):.4f}")
print(f"MSE     : Manual={mse_val:.4f} | sklearn={mean_squared_error(y_act, y_prd):.4f}")
print(f"RMSE    : Manual={rmse_val:.4f} | sklearn={np.sqrt(mean_squared_error(y_act, y_prd)):.4f}")
print(f"R^2     : Manual={r2_val:.4f} | sklearn={r2_score(y_act, y_prd):.4f}")
print(f"Adj R^2 : Manual={adj_r2_val:.4f}")


## 6. Regularization Techniques ($L_1$, $L_2$, ElasticNet)

### Comparison Table

| Model | Penalty Type | Sparsity (Feature Selection) | Best For |
| :--- | :--- | :--- | :--- |
| **Ridge Regression** | $L_2: \lambda \|\beta\|_2^2$ | No (shrinks coefficients) | Multicollinearity & many features. |
| **Lasso Regression** | $L_1: \lambda \|\beta\|_1$ | Yes (exact zeros) | High-dimensional feature selection. |
| **ElasticNet** | $\alpha L_1 + (1-\alpha)L_2$ | Yes | Correlated feature groups. |

### A. Ridge Regression ($L_2$)
$$J_{\text{Ridge}}(\boldsymbol{\beta}) = \frac{1}{2n} \|\mathbf{X}\boldsymbol{\beta} - \mathbf{y}\|^2_2 + \lambda \|\boldsymbol{\beta}\|^2_2$$

Closed-Form Solution:
$$\boldsymbol{\hat{\beta}}_{\text{Ridge}} = (\mathbf{X}^T \mathbf{X} + \lambda \mathbf{I})^{-1} \mathbf{X}^T \mathbf{y}$$

### B. Lasso Regression ($L_1$)
$$J_{\text{Lasso}}(\boldsymbol{\beta}) = \frac{1}{2n} \|\mathbf{X}\boldsymbol{\beta} - \mathbf{y}\|^2_2 + \lambda \|\boldsymbol{\beta}\|_1$$

### C. ElasticNet Regression
$$J_{\text{ElasticNet}}(\boldsymbol{\beta}) = \frac{1}{2n} \|\mathbf{X}\boldsymbol{\beta} - \mathbf{y}\|^2_2 + \lambda \left( r \|\boldsymbol{\beta}\|_1 + \frac{1 - r}{2} \|\boldsymbol{\beta}\|^2_2 \right)$$


In [ ]:
# Example 6.1: Lasso Feature Selection Demo (Setting Unimportant Weights to 0)
X_sparse, y_sparse = make_regression(n_samples=100, n_features=10, n_informative=2, noise=5, random_state=42)

scaler = StandardScaler()
X_sparse_scaled = scaler.fit_transform(X_sparse)

ols = LinearRegression().fit(X_sparse_scaled, y_sparse)
ridge = Ridge(alpha=10.0).fit(X_sparse_scaled, y_sparse)
lasso = Lasso(alpha=2.0).fit(X_sparse_scaled, y_sparse)

df_coefs = pd.DataFrame({
    "Feature": [f"Feature_{i}" for i in range(10)],
    "OLS Coef": ols.coef_.round(2),
    "Ridge Coef (L2)": ridge.coef_.round(2),
    "Lasso Coef (L1)": lasso.coef_.round(2)
})

print("=== REGULARIZATION FEATURE SELECTION (LASSO SPARSITY) ===")
print(df_coefs.to_string(index=False))


## 7. Feature Engineering & Preprocessing

1. **Feature Scaling (Mandatory for GD & Regularization)**:
   - **Standardization (Z-score)**: $x' = \frac{x - \mu}{\sigma}$. Required for Ridge/Lasso penalties to treat features equally.
   - **Min-Max Scaling**: $x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$.
2. **Handling Categorical Features**:
   - **One-Hot Encoding**: Converts categorical levels into binary columns.
   - **Dummy Variable Trap**: Avoid multi-collinearity by dropping one level ($k-1$ dummy columns created for $k$ categories).
3. **Polynomial Features**:
   - Models non-linear curves using linear parameters: $y = \beta_0 + \beta_1 x + \beta_2 x^2$.


In [ ]:
# Example 7.1: Dummy Variable Trap Demonstration
df_cat = pd.DataFrame({"Department": ["Sales", "Engineering", "Marketing", "Sales", "Engineering"]})

# 1. Without dropping first category (Multi-collinearity hazard: sum across rows = 1)
df_all_dummies = pd.get_dummies(df_cat["Department"], drop_first=False, dtype=int)

# 2. Dropping first category (k-1 rule: avoids Dummy Variable Trap)
df_k_minus_1 = pd.get_dummies(df_cat["Department"], drop_first=True, dtype=int)

print("=== FULL DUMMY ENCODING (Trap: Columns sum to 1) ===")
print(df_all_dummies)
print("
=== CORRECT ENCODING (k-1 Dummies: Avoids Trap) ===")
print(df_k_minus_1)
